### 1. Загрузите данные из файла abalone.csv.
Это датасет,в котором требуется предсказать возраст ракушки (число колец) по физическим
измерениям.

In [1]:
import pandas as pd

df = pd.read_csv('abalone.csv')
df

,Sex,Length,Diameter,Height,WholeWeight,ShuckedWeight,VisceraWeight,ShellWeight,Rings
0,M,0.455,0.365,0.095,0.5140,0.2245,0.1010,0.1500,15
1,M,0.350,0.265,0.090,0.2255,0.0995,0.0485,0.0700,7
2,F,0.530,0.420,0.135,0.6770,0.2565,0.1415,0.2100,9
3,M,0.440,0.365,0.125,0.5160,0.2155,0.1140,0.1550,10
4,I,0.330,0.255,0.080,0.2050,0.0895,0.0395,0.0550,7
...,...,...,...,...,...,...,...,...,...
4172,F,0.565,0.450,0.165,0.8870,0.3700,0.2390,0.2490,11
4173,M,0.590,0.440,0.135,0.9660,0.4390,0.2145,0.2605,10
4174,M,0.600,0.475,0.205,1.1760,0.5255,0.2875,0.3080,9
4175,F,0.625,0.485,0.150,1.0945,0.5310,0.2610,0.2960,10


### 2. Преобразуйте признак Sex в числовой: 
значение F должно перейти в -1, I — в 0, M — в 1. 
Если вы используете Pandas, то подойдет
следующий код: 
```
data[’Sex’] = data[’Sex’].map(lambda x: 1 if x == ’M’ else (-1 if x == ’F’ else 0))
```

In [2]:
df['Sex'] = df['Sex'].map(lambda x: 1 if x == 'M' else (-1 if x == 'F' else 0))

### 3. Разделите содержимое файлов на признаки и целевую переменную.
В последнем столбце записана целевая переменная, в остальных —
признаки.

In [3]:
X = df.iloc[:, :-1]
y = df.iloc[:, -1]
X

,Sex,Length,Diameter,Height,WholeWeight,ShuckedWeight,VisceraWeight,ShellWeight
0,1,0.455,0.365,0.095,0.5140,0.2245,0.1010,0.1500
1,1,0.350,0.265,0.090,0.2255,0.0995,0.0485,0.0700
2,-1,0.530,0.420,0.135,0.6770,0.2565,0.1415,0.2100
3,1,0.440,0.365,0.125,0.5160,0.2155,0.1140,0.1550
4,0,0.330,0.255,0.080,0.2050,0.0895,0.0395,0.0550
...,...,...,...,...,...,...,...,...
4172,-1,0.565,0.450,0.165,0.8870,0.3700,0.2390,0.2490
4173,1,0.590,0.440,0.135,0.9660,0.4390,0.2145,0.2605
4174,1,0.600,0.475,0.205,1.1760,0.5255,0.2875,0.3080
4175,-1,0.625,0.485,0.150,1.0945,0.5310,0.2610,0.2960


### 4. Обучите случайный лес (sklearn.ensemble.RandomForestRegressor)
с различным числом деревьев: от 1 до 50 `(random_state=1)`. Для
каждого из вариантов оцените качество работы полученного леса
на кросс-валидации по 5 блокам. Используйте параметры
`"random_state=1"и "shuﬄe=True"` при создании генератора кросс-
валидации `sklearn.cross_validation.KFold`.
В качестве меры качества воспользуйтесь коэффициентом детерминации `(sklearn.metrics.r2_score)`.

In [4]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import KFold, GridSearchCV
from sklearn.metrics import r2_score

rfr = RandomForestRegressor(random_state = 1)
kfold = KFold(n_splits = 5, random_state = 1, shuffle = True)
params_grid = {
    'n_estimators': list(range(1,51))
}
grid_search = GridSearchCV(estimator = rfr, param_grid = params_grid, scoring = 'r2', cv = kfold, n_jobs = -1)
grid_search.fit(X,y)

,estimator,RandomForestR...andom_state=1)
,param_grid,"{'n_estimators': [1, 2, ...]}"
,scoring,'r2'
,n_jobs,-1
,refit,True
,cv,KFold(n_split... shuffle=True)
,verbose,0
,pre_dispatch,'2*n_jobs'
,error_score,nan
,return_train_score,False
,n_estimators,50


In [5]:
grid_search.cv_results_['mean_test_score'].shape

(50,)

### 5. Определите, при каком минимальном количестве деревьев случайный лес показывает качество на кросс-валидации выше 0.52. 
Это количество и будет ответом на задание.

In [6]:
import numpy as np
np.where(grid_search.cv_results_['mean_test_score']> 0.52)[0].min()+1

21

### 6. Обратите внимание на изменение качества по мере роста числа деревьев. Ухудшается ли оно?

In [7]:
r2_scores = grid_search.cv_results_['mean_test_score']
ans = pd.DataFrame({
    'r2_scores': r2_scores,
    'diff with previous': np.insert(np.diff(r2_scores), 0, 0),
    'increases?': ((np.sign(np.insert(np.diff(r2_scores), 0, 0))+1)/2).astype('bool')
})
print(ans)
print('Не ухудшается')

    r2_scores  diff with previous  increases?
0    0.109675            0.000000        True
1    0.341300            0.231625        True
2    0.406434            0.065134        True
3    0.444775            0.038341        True
4    0.465032            0.020258        True
5    0.471396            0.006364        True
6    0.476666            0.005270        True
7    0.482935            0.006269        True
8    0.489437            0.006502        True
9    0.495409            0.005972        True
10   0.494411           -0.000997       False
11   0.499028            0.004617        True
12   0.503058            0.004030        True
13   0.507317            0.004259        True
14   0.509181            0.001864        True
15   0.511411            0.002230        True
16   0.514892            0.003481        True
17   0.517220            0.002329        True
18   0.519829            0.002609        True
19   0.519484           -0.000345       False
20   0.520529            0.001045 